# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Hardik144/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Finding 1 — Growing vs declining content

The paper reports that growing pages were younger on average than declining pages, while their average word counts were similar. My methodology question is how the growth label and comparison windows were defined, and whether the analysis accounts for differences in page visibility and supports the age relationship under time-aware validation.

Finding 2 — Days visible as a growth predictor

The paper identifies days visible as the highest-ranked growth predictor in its machine-learning appendix. My methodology question is whether this feature was measured entirely before the outcome window, and whether the result remains consistent under client-disjoint or time-aware validation.

In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### 2. My model under an honest split (before/after)

In Week 5, I evaluated a Random Forest regression model against a previous-period clicks baseline using a random 80/20 holdout split.

For this validation audit, I compare that evaluation with a grouped split using `client_id`. Grouping helps prevent the same identifiable client from appearing in both the training and testing sets, reducing the risk that client-specific patterns make the evaluation appear more generalizable than it is.

I evaluate both the Random Forest and the baseline using MAE and RMSE on the same test observations within each split. I report the before-and-after results without assuming that a lower error necessarily means the model will generalize to future periods or unseen clients.

The grouped evaluation is a validation sensitivity check. Its results are interpreted alongside the split design, feature availability, and limitations of the dataset.


In [12]:

# SECTION 2: MY MODEL UNDER AN HONEST SPLIT (BEFORE/AFTER)

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error

# --------------------------------------------------
# 1. Verify the dataset
# --------------------------------------------------

if "df" not in globals():
    raise NameError(
        "df is not defined. Run the dataset-loading cell first."
    )

target_col = "clicks_last_30d"
baseline_col = "clicks_prev_30d"

required = [target_col, baseline_col]

missing = [col for col in required if col not in df.columns]

if missing:
    raise ValueError(f"Missing required columns: {missing}")

# Use the Week 5 feature list from the earlier model.
candidate_features = [
    "clicks_prev_30d",
    "impressions_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "search_volume",
    "competition",
    "cp",
    "content_type",
    "main_intent"
]

features = [
    col for col in candidate_features
    if col in df.columns
]

if not features:
    raise ValueError("No eligible Week 5 features were found.")

if target_col in features:
    raise ValueError("The target cannot be included as a feature.")

print("Target:", target_col)
print("Baseline:", baseline_col)
print("Features:", features)

# --------------------------------------------------
# 2. Prepare data
# --------------------------------------------------

model_df = df.copy()

model_df[target_col] = pd.to_numeric(
    model_df[target_col], errors="coerce"
)

model_df = model_df.dropna(subset=[target_col]).copy()

# --------------------------------------------------
# 3. Create BEFORE split: random 80/20 holdout
# --------------------------------------------------

random_train, random_test = train_test_split(
    model_df,
    test_size=0.20,
    random_state=42
)

print("\nBEFORE: RANDOM SPLIT")
print("Training rows:", len(random_train))
print("Testing rows:", len(random_test))

# --------------------------------------------------
# 4. Create AFTER split: grouped by client_id
# --------------------------------------------------

if "client_id" not in model_df.columns:
    raise ValueError(
        "client_id is unavailable. A grouped client split "
        "cannot be performed with this dataset."
    )

# Treat missing client IDs as separate unknown groups.
groups = model_df["client_id"].astype("string").copy()

missing_id = groups.isna()

groups.loc[missing_id] = [
    f"UNKNOWN_ROW_{i}"
    for i in model_df.index[missing_id]
]

group_splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

group_train_idx, group_test_idx = next(
    group_splitter.split(model_df, groups=groups)
)

group_train = model_df.iloc[group_train_idx].copy()
group_test = model_df.iloc[group_test_idx].copy()

print("\nAFTER: GROUPED SPLIT")
print("Training rows:", len(group_train))
print("Testing rows:", len(group_test))

# Verify no identifiable client overlap.
train_clients = set(group_train["client_id"].dropna())
test_clients = set(group_test["client_id"].dropna())

client_overlap = train_clients.intersection(test_clients)

print("Identifiable client overlap:", len(client_overlap))

assert len(client_overlap) == 0, (
    "Grouped split failed: identifiable clients overlap."
)

# --------------------------------------------------
# 5. Define a function to train and evaluate a model
# --------------------------------------------------

def evaluate_split(train_data, test_data, split_name):

    X_train = train_data[features].copy()
    X_test = test_data[features].copy()

    y_train = pd.to_numeric(
        train_data[target_col], errors="coerce"
    ).astype(float)

    y_test = pd.to_numeric(
        test_data[target_col], errors="coerce"
    ).astype(float)

    # Detect numeric and categorical features.
    numeric_features = X_train.select_dtypes(
        include=["number", "bool"]
    ).columns.tolist()

    categorical_features = [
        col for col in features
        if col not in numeric_features
    ]

    # Preprocessing fitted only on the training data.
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median"))
    ])

    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer([
        ("numeric", numeric_pipeline, numeric_features),
        ("categorical", categorical_pipeline, categorical_features)
    ])

    # Week 5 Random Forest configuration.
    rf_model = Pipeline([
        ("preprocessor", preprocessor),
        ("regressor", RandomForestRegressor(
            n_estimators=150,
            max_depth=10,
            min_samples_leaf=5,
            random_state=42,
            n_jobs=-1
        ))
    ])

    # Train the model.
    rf_model.fit(X_train, y_train)

    # Predict on the held-out test observations.
    model_predictions = np.maximum(
        rf_model.predict(X_test), 0
    )

    # Previous-period clicks baseline.
    baseline_predictions = np.maximum(
        pd.to_numeric(
            test_data[baseline_col], errors="coerce"
        ).fillna(0).to_numpy(dtype=float),
        0
    )

    actual = y_test.to_numpy(dtype=float)

    # Calculate metrics on the same test rows.
    model_mae = mean_absolute_error(
        actual, model_predictions
    )

    model_rmse = np.sqrt(
        mean_squared_error(actual, model_predictions)
    )

    baseline_mae = mean_absolute_error(
        actual, baseline_predictions
    )

    baseline_rmse = np.sqrt(
        mean_squared_error(actual, baseline_predictions)
    )

    results = [
        {
            "split": split_name,
            "method": "Previous-period baseline",
            "MAE": baseline_mae,
            "RMSE": baseline_rmse,
            "test_rows": len(test_data)
        },
        {
            "split": split_name,
            "method": "Random Forest",
            "MAE": model_mae,
            "RMSE": model_rmse,
            "test_rows": len(test_data)
        }
    ]

    predictions_df = pd.DataFrame({
        "actual_clicks": actual,
        "baseline_prediction": baseline_predictions,
        "model_prediction": model_predictions
    })

    predictions_df["model_absolute_error"] = np.abs(
        predictions_df["actual_clicks"]
        - predictions_df["model_prediction"]
    )

    predictions_df["baseline_absolute_error"] = np.abs(
        predictions_df["actual_clicks"]
        - predictions_df["baseline_prediction"]
    )

    return results, predictions_df


# --------------------------------------------------
# 6. Evaluate BEFORE and AFTER
# --------------------------------------------------

random_results, random_predictions = evaluate_split(
    random_train,
    random_test,
    "Before: Random 80/20"
)

group_results, group_predictions = evaluate_split(
    group_train,
    group_test,
    "After: Grouped by client_id"
)

# --------------------------------------------------
# 7. Display the before/after comparison
# --------------------------------------------------

comparison_df = pd.DataFrame(
    random_results + group_results
)

print("\nBEFORE / AFTER VALIDATION COMPARISON")
display(comparison_df)

# --------------------------------------------------
# 8. Show the change in Random Forest MAE
# --------------------------------------------------

random_model_mae = comparison_df.loc[
    (comparison_df["split"] == "Before: Random 80/20") &
    (comparison_df["method"] == "Random Forest"),
    "MAE"
].iloc[0]

group_model_mae = comparison_df.loc[
    (comparison_df["split"] == "After: Grouped by client_id") &
    (comparison_df["method"] == "Random Forest"),
    "MAE"
].iloc[0]

print("\nRANDOM FOREST MAE CHANGE")
print("Random split MAE:", round(random_model_mae, 4))
print("Grouped split MAE:", round(group_model_mae, 4))
print(
    "Difference (grouped - random):",
    round(group_model_mae - random_model_mae, 4)
)

print("\nValidation comparison completed.")

Target: clicks_last_30d
Baseline: clicks_prev_30d
Features: ['clicks_prev_30d', 'impressions_prev_30d', 'sessions_prev_30d', 'content_age_days', 'search_volume', 'competition', 'content_type', 'main_intent']

BEFORE: RANDOM SPLIT
Training rows: 24000
Testing rows: 6000

AFTER: GROUPED SPLIT
Training rows: 23837
Testing rows: 6163
Identifiable client overlap: 0

BEFORE / AFTER VALIDATION COMPARISON


,split,method,MAE,RMSE,test_rows
0,Before: Random 80/20,Previous-period baseline,2.269167,9.929997,6000
1,Before: Random 80/20,Random Forest,2.108391,8.376439,6000
2,After: Grouped by client_id,Previous-period baseline,1.897939,10.267310,6163
3,After: Grouped by client_id,Random Forest,1.909690,12.609042,6163



RANDOM FOREST MAE CHANGE
Random split MAE: 2.1084
Grouped split MAE: 1.9097
Difference (grouped - random): -0.1987

Validation comparison completed.


## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### 3. Leakage audit

I audited the features used in my Week 5 Random Forest model to identify possible target leakage.

The prediction target is `clicks_last_30d`. The model uses previous-period performance, content characteristics, and search-related features. I checked for direct target inclusion, suspicious feature names, and unusually strong numerical relationships with the target.

A feature with a strong correlation is not automatically leakage. The important question is whether that feature was genuinely available at prediction time, before the target outcome window began.

This audit identifies potential risks that require checking against the dataset's data contract and measurement windows. I will not describe the model as leakage-free unless the feature definitions and timing support that conclusion.


In [13]:

# SECTION 3: LEAKAGE AUDIT

import pandas as pd
import numpy as np

# --------------------------------------------------
# 1. Verify that the dataset and Week 5 features exist
# --------------------------------------------------

if "df" not in globals():
    raise NameError(
        "df is not defined. Run the dataset-loading cell first."
    )

target_col = "clicks_last_30d"

if target_col not in df.columns:
    raise ValueError(f"Missing target column: {target_col}")

# Reuse the features from Section 2, if available.
if "features" in globals():
    audit_features = [
        col for col in features
        if col in df.columns
    ]
else:
    candidate_features = [
        "clicks_prev_30d",
        "impressions_prev_30d",
        "sessions_prev_30d",
        "content_age_days",
        "search_volume",
        "competition",
        "cp",
        "content_type",
        "main_intent"
    ]

    audit_features = [
        col for col in candidate_features
        if col in df.columns
    ]

if not audit_features:
    raise ValueError("No eligible Week 5 features were found.")

print("Target:", target_col)
print("Features audited:", audit_features)

# --------------------------------------------------
# 2. Direct target leakage check
# --------------------------------------------------

print("\n1. DIRECT TARGET CHECK")

if target_col in audit_features:
    print("WARNING: The target is included in the feature list.")
else:
    print("The target is not directly included in the feature list.")

# --------------------------------------------------
# 3. Check feature names for possible leakage indicators
# --------------------------------------------------

suspicious_terms = [
    "target",
    "label",
    "outcome",
    "future",
    "next_period",
    "next_30",
    "last_30",
    "current_30",
    "post_period",
    "after"
]

name_audit_rows = []

for col in audit_features:

    matched_terms = [
        term for term in suspicious_terms
        if term in col.lower()
    ]

    name_audit_rows.append({
        "feature": col,
        "data_type": str(df[col].dtype),
        "possible_name_flag": (
            ", ".join(matched_terms)
            if matched_terms
            else "No name-based flag"
        )
    })

name_audit_df = pd.DataFrame(name_audit_rows)

print("\n2. FEATURE NAME AUDIT")
display(name_audit_df)

# --------------------------------------------------
# 4. Identify columns that may describe the outcome period
# --------------------------------------------------

window_terms = [
    "date",
    "time",
    "week",
    "month",
    "period",
    "last_30",
    "prev_30",
    "90d",
    "future"
]

window_columns = [
    col for col in df.columns
    if any(term in col.lower() for term in window_terms)
]

print("\n3. POSSIBLE TIME/WINDOW COLUMNS")
print(window_columns)

print(
    "\nReview the dataset documentation to establish "
    "when each feature was measured."
)

# --------------------------------------------------
# 5. Numerical correlations with the target
# --------------------------------------------------

numeric_features = [
    col for col in audit_features
    if pd.api.types.is_numeric_dtype(df[col])
]

if numeric_features:

    corr_data = df[numeric_features + [target_col]].copy()

    for col in corr_data.columns:
        corr_data[col] = pd.to_numeric(
            corr_data[col], errors="coerce"
        )

    correlations = (
        corr_data.corr(numeric_only=True)[target_col]
        .drop(target_col, errors="ignore")
        .sort_values(
            key=lambda values: values.abs(),
            ascending=False
        )
    )

    correlation_df = correlations.rename(
        "correlation_with_target"
    ).reset_index()

    correlation_df = correlation_df.rename(
        columns={"index": "feature"}
    )

    print("\n4. NUMERICAL CORRELATION CHECK")
    display(correlation_df)

else:
    print("\nNo numerical features available for correlation.")

# --------------------------------------------------
# 6. Check for exact equality with the target
# --------------------------------------------------

print("\n5. EXACT TARGET MATCH CHECK")

target_values = pd.to_numeric(
    df[target_col], errors="coerce"
)

exact_matches = []

for col in numeric_features:

    feature_values = pd.to_numeric(
        df[col], errors="coerce"
    )

    valid = target_values.notna() & feature_values.notna()

    if valid.sum() > 0:

        if np.array_equal(
            target_values[valid].to_numpy(),
            feature_values[valid].to_numpy()
        ):
            exact_matches.append(col)

if exact_matches:
    print("WARNING: Exact target matches found:", exact_matches)
else:
    print("No exact numerical target matches found.")

# --------------------------------------------------
# 7. Produce a concise audit summary
# --------------------------------------------------

print("\n6. AUDIT SUMMARY")

print("Total features audited:", len(audit_features))

print(
    "Features with suspicious names:",
    int(
        (name_audit_df["possible_name_flag"] !=
         "No name-based flag").sum()
    )
)

print("Exact numerical target matches:", exact_matches)

print(
    "\nAudit completed. Correlations and name flags are "
    "screening checks, not proof of leakage."
)

print(
    "Confirm feature availability and measurement windows "
    "against the dataset documentation before making "
    "a final leakage-free claim."
)

Target: clicks_last_30d
Features audited: ['clicks_prev_30d', 'impressions_prev_30d', 'sessions_prev_30d', 'content_age_days', 'search_volume', 'competition', 'content_type', 'main_intent']

1. DIRECT TARGET CHECK
The target is not directly included in the feature list.

2. FEATURE NAME AUDIT


,feature,data_type,possible_name_flag
0,clicks_prev_30d,int64,No name-based flag
1,impressions_prev_30d,int64,No name-based flag
2,sessions_prev_30d,int64,No name-based flag
3,content_age_days,int64,No name-based flag
4,search_volume,float64,No name-based flag
5,competition,float64,No name-based flag
6,content_type,object,No name-based flag
7,main_intent,object,No name-based flag



3. POSSIBLE TIME/WINDOW COLUMNS
['impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'days_since_last_update']

Review the dataset documentation to establish when each feature was measured.

4. NUMERICAL CORRELATION CHECK


,feature,correlation_with_target
0,clicks_prev_30d,0.922519
1,impressions_prev_30d,0.677568
2,sessions_prev_30d,0.651884
3,competition,-0.031902
4,content_age_days,-0.024262
5,search_volume,-0.009635



5. EXACT TARGET MATCH CHECK
No exact numerical target matches found.

6. AUDIT SUMMARY
Total features audited: 8
Features with suspicious names: 0
Exact numerical target matches: []

Audit completed. Correlations and name flags are screening checks, not proof of leakage.
Confirm feature availability and measurement windows against the dataset documentation before making a final leakage-free claim.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

In [14]:

# SECTION 4: CLAIM REWRITE

import numpy as np
import pandas as pd

print("SECTION 4: CLAIM REWRITE")
print("-" * 60)

# Check whether Section 3 has executed successfully
required_variables = [
    "model_mae",
    "baseline_mae",
    "model_rmse",
    "baseline_rmse"
]

missing_variables = [
    name for name in required_variables
    if name not in globals()
]

if missing_variables:
    print("ERROR: Some Section 3 variables are missing:")
    print(missing_variables)
    print("\nPlease rerun Section 3 successfully before running Section 4.")

else:
    # Display the actual measured metrics
    print("\n1. MEASURED RESULTS")

    print(f"Baseline MAE: {baseline_mae:.4f}")
    print(f"Random Forest MAE: {model_mae:.4f}")
    print(f"Baseline RMSE: {baseline_rmse:.4f}")
    print(f"Random Forest RMSE: {model_rmse:.4f}")

    # Calculate relative MAE change
    if baseline_mae != 0:
        relative_change = (
            (model_mae - baseline_mae) / baseline_mae
        ) * 100

        print(f"Relative MAE change: {relative_change:.2f}%")

    else:
        relative_change = None
        print("Relative MAE change: undefined (baseline MAE is zero)")

    # Interpret the measured comparison
    print("\n2. MODEL VS BASELINE")

    if model_mae < baseline_mae:
        print(
            "The Random Forest achieved lower MAE than the "
            "baseline on the evaluated test set."
        )

    elif model_mae > baseline_mae:
        print(
            "The Random Forest achieved higher MAE than the "
            "baseline on the evaluated test set."
        )

    else:
        print(
            "The Random Forest and baseline achieved equal MAE "
            "on the evaluated test set."
        )

    # Write the research claim using cautious language
    print("\n3. REWRITTEN RESEARCH CLAIM")

    print(
        "The Random Forest model was evaluated for predicting "
        "clicks_last_30d using historical content and search-related "
        "features. Its performance was compared with a baseline "
        "based on clicks_prev_30d. The measured MAE and RMSE describe "
        "prediction errors on the held-out evaluation data. These "
        "results provide evidence about predictive performance under "
        "the tested validation design. They do not establish causation, "
        "prove Google's ranking algorithm, or guarantee future results."
    )

    # Limitations
    print("\n4. LIMITATIONS")

    print(
        "The findings are limited to the dataset, selected features, "
        "model configuration, and validation design used in this "
        "experiment. The results should be treated as decision-support "
        "evidence rather than a guarantee of future clicks or search "
        "visibility. Further validation on new data would be needed "
        "before making broader claims."
    )

    print("\nSection 4 completed.")

SECTION 4: CLAIM REWRITE
------------------------------------------------------------
ERROR: Some Section 3 variables are missing:
['model_mae', 'baseline_mae', 'model_rmse', 'baseline_rmse']

Please rerun Section 3 successfully before running Section 4.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.